# Dataset generation


## Qu'est ce que ca fait?
------------------------------------------------------------------
Construit le dataset du projet "Can an AI Recognize Another AI?"

Deux moitiés, un seul CSV en sortie :
  (1) TEXTES IA  : pour chaque prompt x chaque modele open-weight -> generation
  (2) TEXTES HUMAINS : echantillonnes depuis des corpus existants, tronques

Format de sortie (une ligne = un texte) :
  id, domain, prompt_id, source_type, model_name, text, n_words, split

Lancer de preference sur GPU (Colab). Les modeles open-weight se telechargent
automatiquement depuis HuggingFace ; certains (gemma, mistral) demandent
d'accepter la licence sur le site + un token HF (huggingface-cli login).

Dependances :
  pip install torch transformers datasets pandas
------------------------------------------------------------------



Il faut cree un compte hugging face, se connecter et accepter les parametres d'utilisation de certain models comme gemma 3 ou 4 par exemple.

Un autre probleme qui est survenu avec gemma est qu'il s'agit d'un model multimodal donc la seule utilisation de automodel ne suffit pas.

Mistral est casse couille a faire marcher donc pour l'instant on l'evite.

In [1]:
!pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    datasets \
    huggingface_hub \
    sentencepiece \
    safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.9 MB/s eta 0:00:00


In [2]:
import os
import re
import gc
import json
import random
import traceback
from pathlib import Path
from typing import Any

import pandas as pd
import torch

from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    set_seed,
)

In [3]:
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/Final_Project_NLP"

# Check that the folder exists
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(
        f"Folder not found: {PROJECT_DIR}\n"
        "Check the folder name and its location in Google Drive."
    )

# Make it the current working directory
os.chdir(PROJECT_DIR)

print("Current working directory:", os.getcwd())
print("Files in the folder:")
print(os.listdir("."))

Mounted at /content/drive
Current working directory: /content/drive/.shortcut-targets-by-id/1TOIW12vmjPTJ4-9tHQ4CPzftb-so-rz5/Final_Project_NLP
Files in the folder:
['NLP_Project_Proposals.docx', 'NLP_Project_Plan_and_Dataset.md', 'prompts.csv', 'human_corpora_sources.md', 'build_dataset.py', '__pycache__', 'detector.py', 'classifier_attribution.py', 'llm_judge.py', 'human_baseline.py', 'Fil Conducteur.gdoc', 'dataset_de_merde.csv', 'generation_checkpoints', '.ipynb_checkpoints', 'dataset.csv', 'Dataset_generation_NLP.ipynb']


In [4]:
from huggingface_hub import notebook_login
notebook_login()

In [5]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Select Runtime → Change runtime type → GPU."
    )

GPU_NAME = torch.cuda.get_device_name(0)

print("\nGPU:", GPU_NAME)
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)


GPU: Tesla T4
GPU memory: 14.56 GB


## Configuration


In [6]:
#PROMPTS_CSV = "prompts.csv"          # les 25 prompts (meme dossier)
#OUTPUT_CSV  = "dataset.csv"          # le dataset final
PROJECT_DIR = Path("/content/drive/MyDrive/Final_Project_NLP")
PROMPTS_CSV = PROJECT_DIR / "prompts.csv"
OUTPUT_CSV = PROJECT_DIR / "dataset.csv"
QUICK_TEST_OUTPUT_CSV = PROJECT_DIR / "quick_test_dataset.csv"

CHECKPOINT_DIR = PROJECT_DIR / "generation_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


MODEL_CSV = PROJECT_DIR / "dataset.csv"
HUMAN_CSV = PROJECT_DIR / "human_texts.csv"
COMBINED_CSV = PROJECT_DIR / "dataset_with_humans.csv"

# Change this name whenever generation settings change.
# This prevents old checkpoints from mixing with new settings.
RUN_NAME = "open_models_gemma4_v1"

# Modeles open-weight a faire generer. Prenez de PETITES versions.
# NB : gemma / mistral sont "gated" -> acceptez la licence sur HF + login.
OPEN_MODELS = [
    "Qwen/Qwen3-4B-Instruct-2507",
    "google/gemma-4-E4B-it",
    "microsoft/Phi-4-mini-instruct",
    "HuggingFaceTB/SmolLM3-3B",
    "mistralai/Mistral-7B-Instruct-v0.3"
]

'''WORD_CAP        = 350     # on tronque tous les textes a ~350 mots
MAX_NEW_TOKENS  = 500     # ~350 mots ~= 450-500 tokens
TEMPERATURE     = 0.8
N_HUMAN_PER_DOMAIN = 40    # nb de textes humains a garder par domaine
MIN_WORDS_HUMAN = 150      # on ignore les textes humains trop courts'''

MIN_WORDS = 300
MAX_WORDS = 400

MIN_WORDS_HUMAN = 300
MAX_WORDS_HUMAN = 400

# 650 tokens normally gives enough room for 300–400 words.
MAX_NEW_TOKENS = 650
MAX_ATTEMPTS = 3

TEMPERATURE = 0.8
TOP_P = 0.95
REPETITION_PENALTY = 1.05

BASE_SEED = 42
RANDOM_SEED = 42

# Use 4-bit loading for Colab.
USE_4BIT = True

# Chemin local du CSV PERSUADE / DAIGT (train_essays.csv de Kaggle) pour les
# essais d'opinion. Laissez None si vous ne l'avez pas -> le domaine est saute.
PERSUADE_CSV = None        # ex: "train_essays.csv"

QUICK_TEST = False         # True = seulement 2 prompts x 1 modele (pour tester vite)

REQUIRE_ALL_MODELS = True

# Automatically resume completed prompts after interruption.
RESUME_FROM_CHECKPOINTS = True

# Human samples are skipped during the model quick test.
INCLUDE_HUMANS_IN_FULL_RUN = True

# Human candidate-selection settings.
HUMAN_CANDIDATE_MIN_WORDS = 150
HUMAN_CANDIDATE_MAX_WORDS = 600
HUMAN_TARGET_WORDS = 350
HUMAN_MAX_FINAL_WORDS = 400
HUMAN_POOL_SIZE = 300
HUMAN_MAX_EXAMPLES_SCANNED = 100_000

print("Models:")
for model_id in OPEN_MODELS:
    print("-", model_id)


Models:
- Qwen/Qwen3-4B-Instruct-2507
- google/gemma-4-E4B-it
- microsoft/Phi-4-mini-instruct
- HuggingFaceTB/SmolLM3-3B
- mistralai/Mistral-7B-Instruct-v0.3


In [7]:
!nvidia-smi

Mon Jul 27 13:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Utils

In [8]:
# ---------------------------------------------------------
# Precision and quantization
# ---------------------------------------------------------

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

if USE_4BIT:
    QUANTIZATION_CONFIG = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
else:
    QUANTIZATION_CONFIG = None

print("Compute dtype:", COMPUTE_DTYPE)
print("4-bit loading:", USE_4BIT)
# =========================================================
# 5. Small utility functions
# =========================================================

def count_words(text):
    return len(str(text).split())


def clean_text(text):
    return " ".join(str(text).split()).strip()


def truncate_text(text, max_words=MAX_WORDS):
    words = text.split()
    return " ".join(words[:max_words])


def n_words(text: str) -> int:
    return len(str(text).split())


def clean_text_human(text: str) -> str:
    text = str(text)

    # Remove common WritingPrompts tags.
    text = re.sub(
        r"\[\s*(WP|RF|TT|IP|FF|OT|CW|EU|PI|PM)\s*\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove URLs and basic Markdown symbols.
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\\*", "").replace("*", "")

    # Normalize spaces.
    text = re.sub(r"\s+", " ", text).strip()

    return text


def fit_word_range(
    text: str,
    min_words: int = MIN_WORDS_HUMAN,
    max_words: int = MAX_WORDS_HUMAN,
):
    """
    Return text between min_words and max_words.

    Long texts are cut at a complete sentence.
    Short texts are rejected.
    """

    text = clean_text_human(text)
    word_count = n_words(text)

    if word_count < min_words:
        return None

    if word_count <= max_words:
        return text

    # Keep complete sentences until reaching the limit.
    sentences = re.split(r"(?<=[.!?])\s+", text)

    selected_sentences = []
    selected_words = 0

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        sentence_words = n_words(sentence)

        if selected_words + sentence_words > max_words:
            break

        selected_sentences.append(sentence)
        selected_words += sentence_words

    result = " ".join(selected_sentences).strip()

    if n_words(result) < min_words:
        return None

    return result


def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Compute dtype: torch.bfloat16
4-bit loading: True


## AI Models Text Generation


### Load Model

In [ ]:
'''MODELS = [
    "microsoft/Phi-4-mini-instruct",
]'''

def load_model(model_id):
    print(f"\nLoading {model_id}...")

    common_arguments = {
        "device_map": "auto",
        "quantization_config": QUANTIZATION_CONFIG,
        "low_cpu_mem_usage": True,
    }

    # Gemma 4 uses a multimodal model class.
    if model_id.startswith("google/gemma-4"):
        processor = AutoProcessor.from_pretrained(model_id)

        model = AutoModelForMultimodalLM.from_pretrained(
            model_id,
            **common_arguments,
        ).eval()

        return {
            "type": "gemma4",
            "model": model,
            "tokenizer": processor.tokenizer,
            "processor": processor,
        }

    trust_remote_code = False

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=trust_remote_code,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=trust_remote_code,
        **common_arguments,
    ).eval()

    return {
        "type": "causal",
        "model": model,
        "tokenizer": tokenizer,
        "processor": None,
    }

### Prepare Input

In [ ]:
def prepare_inputs(bundle, model_id, instruction):
    model = bundle["model"]

    messages = [
        {
            "role": "system",
            "content": (
                "Return only the requested text. "
                "Do not show reasoning or discuss the instructions."
            ),
        },
        {
            "role": "user",
            "content": instruction,
        },
    ]

    if bundle["type"] == "gemma4":
        processor = bundle["processor"]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            enable_thinking=False,
        )

    else:
        tokenizer = bundle["tokenizer"]

        template_arguments = {}

        # Disable SmolLM3's visible thinking mode.
        if model_id == "HuggingFaceTB/SmolLM3-3B":
            template_arguments["enable_thinking"] = False

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
            **template_arguments,
        )

    return inputs.to(model.device)


### Texts Generation

In [ ]:
def generate_for_model(model_id: str, prompts: pd.DataFrame) -> list:
    print(f"\n=== Modele : {model_id} ===")
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="auto", low_cpu_mem_usage=True
    ).eval()

    rows = []
    for _, r in prompts.iterrows():
        instruction = r["prompt_text"] + " Write about 350 words."

        # Utilise le chat template du modele si disponible
        if tok.chat_template:
            inputs = tok.apply_chat_template(
                [{"role": "user", "content": instruction}],
                add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)
            input_len = inputs.shape[1]
        else:
            enc = tok(instruction, return_tensors="pt").to(model.device)
            inputs = enc["input_ids"]
            input_len = inputs.shape[1]

        with torch.no_grad():
            out = model.generate(
                inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=0.95,
                pad_token_id=tok.eos_token_id,
            )
        # On decode SEULEMENT les nouveaux tokens (on retire le prompt)
        gen = tok.decode(out[0][input_len:], skip_special_tokens=True)
        gen = truncate_words(clean_text(gen), WORD_CAP)

        rows.append({
            "domain": r["domain"],
            "prompt_id": r["prompt_id"],
            "source_type": "model",
            "model_name": model_id.split("/")[-1],
            "text": gen,
            "n_words": n_words(gen),
        })
        print(f"  {r['prompt_id']} -> {n_words(gen)} mots")

    # Libere la memoire avant le modele suivant
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return rows

In [ ]:
# =========================================================
# 8. Generate one text
# =========================================================

def generate_one_text(
    bundle,
    model_id,
    prompt_text,
    seed,
):
    model = bundle["model"]
    tokenizer = bundle["tokenizer"]

    instruction = (
        f"{prompt_text}\n\n"
        "Write between 300 and 400 words. "
        "Finish with a complete sentence."
    )

    set_seed(seed)

    inputs = prepare_inputs(
        bundle=bundle,
        model_id=model_id,
        instruction=instruction,
    )

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0, input_length:]

    if bundle["type"] == "gemma4":
        text = bundle["processor"].decode(
            generated_tokens,
            skip_special_tokens=True,
        )
    else:
        text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

    text = clean_text(text)

    if count_words(text) > MAX_WORDS:
        text = truncate_text(text)

    return text


# =========================================================
# 9. Generate all requested texts
# =========================================================

def generate_dataset():
    prompts = pd.read_csv(PROMPTS_CSV)

    required_columns = {
        "prompt_id",
        "domain",
        "prompt_text",
    }

    missing_columns = required_columns - set(prompts.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in prompts.csv: {missing_columns}"
        )

    if QUICK_TEST:
        prompts = prompts.head(1)

    # Resume an interrupted run.
    if OUTPUT_CSV.exists():
        existing_df = pd.read_csv(OUTPUT_CSV)
        rows = existing_df.to_dict("records")

        completed = set(
            zip(
                existing_df["model_id"].astype(str),
                existing_df["prompt_id"].astype(str),
            )
        )

        print(
            f"Resuming with {len(rows)} existing rows."
        )
    else:
        rows = []
        completed = set()

    failed_models = []

    for model_id in OPEN_MODELS: # MODELS
        bundle = None

        try:
            bundle = load_model(model_id)

            for prompt_number, (_, prompt) in enumerate(
                prompts.iterrows()
            ):
                prompt_id = str(prompt["prompt_id"])

                if (model_id, prompt_id) in completed:
                    print(
                        f"{model_id} / {prompt_id}: already done"
                    )
                    continue

                selected_text = ""
                selected_seed = None
                valid_length = False

                for attempt in range(MAX_ATTEMPTS):
                    seed = (
                        BASE_SEED
                        + prompt_number * 10
                        + attempt
                    )

                    text = generate_one_text(
                        bundle=bundle,
                        model_id=model_id,
                        prompt_text=str(prompt["prompt_text"]),
                        seed=seed,
                    )

                    word_count = count_words(text)

                    print(
                        f"{model_id} / {prompt_id} / "
                        f"attempt {attempt + 1}: "
                        f"{word_count} words"
                    )

                    selected_text = text
                    selected_seed = seed

                    if MIN_WORDS <= word_count <= MAX_WORDS:
                        valid_length = True
                        break

                row = {
                    "domain": prompt["domain"],
                    "prompt_id": prompt_id,
                    "source_type": "model",
                    "model_name": model_id.split("/")[-1],
                    "model_id": model_id,
                    "text": selected_text,
                    "n_words": count_words(selected_text),
                    "valid_length": valid_length,
                    "seed": selected_seed,
                    "temperature": TEMPERATURE,
                    "top_p": TOP_P,
                }

                rows.append(row)
                completed.add((model_id, prompt_id))

                # Save after every generated text.
                pd.DataFrame(rows).to_csv(
                    OUTPUT_CSV,
                    index=False,
                )

        except Exception as error:
            print("\nMODEL FAILED:", model_id)
            print("Error type:", type(error).__name__)
            print("Error:", repr(error))
            traceback.print_exc()

            failed_models.append(model_id)

        finally:
            if bundle is not None:
                del bundle

            clear_memory()

    dataframe = pd.DataFrame(rows)

    print("\nFinished.")
    print("Saved to:", OUTPUT_CSV)
    print("Rows:", len(dataframe))

    if not dataframe.empty:
        print("\nRows per model:")
        print(dataframe.groupby("model_name").size())

    if failed_models:
        print("\nFailed models:")
        for model_id in failed_models:
            print("-", model_id)

    return dataframe

### Test Model generation

In [ ]:
dataset_df = generate_dataset()

Resuming with 5 existing rows.

Loading Qwen/Qwen3-4B-Instruct-2507...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen/Qwen3-4B-Instruct-2507 / P01: already done
Qwen/Qwen3-4B-Instruct-2507 / P02 / attempt 1: 319 words
Qwen/Qwen3-4B-Instruct-2507 / P03 / attempt 1: 319 words
Qwen/Qwen3-4B-Instruct-2507 / P04 / attempt 1: 296 words
Qwen/Qwen3-4B-Instruct-2507 / P04 / attempt 2: 331 words
Qwen/Qwen3-4B-Instruct-2507 / P05 / attempt 1: 319 words
Qwen/Qwen3-4B-Instruct-2507 / P06 / attempt 1: 285 words
Qwen/Qwen3-4B-Instruct-2507 / P06 / attempt 2: 315 words
Qwen/Qwen3-4B-Instruct-2507 / P07 / attempt 1: 332 words
Qwen/Qwen3-4B-Instruct-2507 / P08 / attempt 1: 283 words
Qwen/Qwen3-4B-Instruct-2507 / P08 / attempt 2: 323 words
Qwen/Qwen3-4B-Instruct-2507 / P09 / attempt 1: 368 words
Qwen/Qwen3-4B-Instruct-2507 / P10 / attempt 1: 352 words
Qwen/Qwen3-4B-Instruct-2507 / P11 / attempt 1: 331 words
Qwen/Qwen3-4B-Instruct-2507 / P12 / attempt 1: 322 words
Qwen/Qwen3-4B-Instruct-2507 / P13 / attempt 1: 296 words
Qwen/Qwen3-4B-Instruct-2507 / P13 / attempt 2: 294 words
Qwen/Qwen3-4B-Instruct-2507 / P13 / atte

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

google/gemma-4-E4B-it / P01: already done
google/gemma-4-E4B-it / P02 / attempt 1: 308 words
google/gemma-4-E4B-it / P03 / attempt 1: 314 words
google/gemma-4-E4B-it / P04 / attempt 1: 312 words
google/gemma-4-E4B-it / P05 / attempt 1: 334 words
google/gemma-4-E4B-it / P06 / attempt 1: 304 words
google/gemma-4-E4B-it / P07 / attempt 1: 339 words
google/gemma-4-E4B-it / P08 / attempt 1: 351 words
google/gemma-4-E4B-it / P09 / attempt 1: 324 words
google/gemma-4-E4B-it / P10 / attempt 1: 372 words
google/gemma-4-E4B-it / P11 / attempt 1: 349 words
google/gemma-4-E4B-it / P12 / attempt 1: 349 words
google/gemma-4-E4B-it / P13 / attempt 1: 339 words
google/gemma-4-E4B-it / P14 / attempt 1: 321 words
google/gemma-4-E4B-it / P15 / attempt 1: 346 words
google/gemma-4-E4B-it / P16 / attempt 1: 349 words
google/gemma-4-E4B-it / P17 / attempt 1: 312 words
google/gemma-4-E4B-it / P18 / attempt 1: 315 words
google/gemma-4-E4B-it / P19 / attempt 1: 290 words
google/gemma-4-E4B-it / P19 / attempt 2:

...

microsoft/Phi-4-mini-instruct / P01: already done
microsoft/Phi-4-mini-instruct / P02 / attempt 1: 243 words
microsoft/Phi-4-mini-instruct / P02 / attempt 2: 400 words
microsoft/Phi-4-mini-instruct / P03 / attempt 1: 400 words
microsoft/Phi-4-mini-instruct / P04 / attempt 1: 400 words
microsoft/Phi-4-mini-instruct / P05 / attempt 1: 331 words
microsoft/Phi-4-mini-instruct / P06 / attempt 1: 378 words
microsoft/Phi-4-mini-instruct / P07 / attempt 1: 379 words
microsoft/Phi-4-mini-instruct / P08 / attempt 1: 366 words
microsoft/Phi-4-mini-instruct / P09 / attempt 1: 363 words
microsoft/Phi-4-mini-instruct / P10 / attempt 1: 354 words
microsoft/Phi-4-mini-instruct / P11 / attempt 1: 400 words
microsoft/Phi-4-mini-instruct / P12 / attempt 1: 399 words
microsoft/Phi-4-mini-instruct / P13 / attempt 1: 299 words
microsoft/Phi-4-mini-instruct / P13 / attempt 2: 338 words
microsoft/Phi-4-mini-instruct / P14 / attempt 1: 400 words
microsoft/Phi-4-mini-instruct / P15 / attempt 1: 400 words
micros

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM3-3B / P01: already done
HuggingFaceTB/SmolLM3-3B / P02 / attempt 1: 293 words
HuggingFaceTB/SmolLM3-3B / P02 / attempt 2: 367 words
HuggingFaceTB/SmolLM3-3B / P03 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P04 / attempt 1: 274 words
HuggingFaceTB/SmolLM3-3B / P04 / attempt 2: 377 words
HuggingFaceTB/SmolLM3-3B / P05 / attempt 1: 335 words
HuggingFaceTB/SmolLM3-3B / P06 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P07 / attempt 1: 376 words
HuggingFaceTB/SmolLM3-3B / P08 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P09 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P10 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P11 / attempt 1: 353 words
HuggingFaceTB/SmolLM3-3B / P12 / attempt 1: 358 words
HuggingFaceTB/SmolLM3-3B / P13 / attempt 1: 384 words
HuggingFaceTB/SmolLM3-3B / P14 / attempt 1: 349 words
HuggingFaceTB/SmolLM3-3B / P15 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P16 / attempt 1: 400 words
HuggingFaceTB/SmolLM3-3B / P17 / atte

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

mistralai/Mistral-7B-Instruct-v0.3 / P01: already done
mistralai/Mistral-7B-Instruct-v0.3 / P02 / attempt 1: 348 words
mistralai/Mistral-7B-Instruct-v0.3 / P03 / attempt 1: 370 words
mistralai/Mistral-7B-Instruct-v0.3 / P04 / attempt 1: 392 words
mistralai/Mistral-7B-Instruct-v0.3 / P05 / attempt 1: 353 words
mistralai/Mistral-7B-Instruct-v0.3 / P06 / attempt 1: 379 words
mistralai/Mistral-7B-Instruct-v0.3 / P07 / attempt 1: 365 words
mistralai/Mistral-7B-Instruct-v0.3 / P08 / attempt 1: 400 words
mistralai/Mistral-7B-Instruct-v0.3 / P09 / attempt 1: 367 words
mistralai/Mistral-7B-Instruct-v0.3 / P10 / attempt 1: 290 words
mistralai/Mistral-7B-Instruct-v0.3 / P10 / attempt 2: 400 words
mistralai/Mistral-7B-Instruct-v0.3 / P11 / attempt 1: 367 words
mistralai/Mistral-7B-Instruct-v0.3 / P12 / attempt 1: 334 words
mistralai/Mistral-7B-Instruct-v0.3 / P13 / attempt 1: 327 words
mistralai/Mistral-7B-Instruct-v0.3 / P14 / attempt 1: 317 words
mistralai/Mistral-7B-Instruct-v0.3 / P15 / attemp

## Human texts

### Load Human Models

In [17]:
def get_human_dataset(domain: str):
    if domain == "news":
        dataset = load_dataset(
            "abisee/cnn_dailymail",
            "3.0.0",
            split="train",
            streaming=True,
        )

        return dataset, "article", "abisee/cnn_dailymail"

    if domain == "review":
        dataset = load_dataset(
            "Yelp/yelp_review_full",
            split="train",
            streaming=True,
        )

        return dataset, "text", "Yelp/yelp_review_full"

    if domain == "story":
        dataset = load_dataset(
            "euclaise/writingprompts",
            split="train",
            streaming=True,
        )

        return dataset, "story", "euclaise/writingprompts"

    if domain == "howto":
        dataset = load_dataset(
            "boundless-asura/wikihow",
            split="train",
            streaming=True,
        )

        return dataset, "text", "boundless-asura/wikihow"

    if domain == "opinion":
        dataset = load_dataset(
            "nlpatunt/D_persuade_2",
            split="train",
            streaming=True,
        )

        return dataset, "full_text", "nlpatunt/D_persuade_2"

    raise ValueError(f"Unknown domain: {domain}")

### Select human text for one domain

In [18]:
def load_human_domain(
    domain: str,
    number_required: int,
) -> list[dict]:
    print(
        f"\n--- Human texts: {domain} "
        f"({number_required} required) ---"
    )

    dataset, text_column, dataset_name = get_human_dataset(domain)

    # Avoid selecting the first examples from the dataset.
    dataset = dataset.shuffle(
        seed=RANDOM_SEED,
        buffer_size=10_000,
    )

    rows = []
    seen_texts = set()

    for example in dataset:
        raw_text = example.get(text_column)

        if not isinstance(raw_text, str):
            continue

        text = fit_word_range(raw_text)

        if text is None:
            continue

        # Avoid exact duplicates.
        normalized_text = text.lower()

        if normalized_text in seen_texts:
            continue

        seen_texts.add(normalized_text)

        rows.append({
            "domain": domain,
            "prompt_id": "",
            "source_type": "human",
            "model_name": "human",
            "model_id": "human",
            "text": text,
            "n_words": n_words(text),
            "valid_length": True,
            "seed": "",
            "temperature": "",
            "top_p": "",
            "human_dataset": dataset_name,
        })

        print(
            f"  Human text {len(rows)}/{number_required}: "
            f"{n_words(text)} words"
        )

        if len(rows) >= number_required:
            break

    if len(rows) < number_required:
        raise RuntimeError(
            f"Only {len(rows)} suitable human texts were found "
            f"for domain '{domain}', but {number_required} were required."
        )

    return rows

### Generate Human Dataset

In [11]:
def create_human_dataset():
    if not MODEL_CSV.exists():
        raise FileNotFoundError(
            f"Model CSV not found: {MODEL_CSV}"
        )

    model_df = pd.read_csv(MODEL_CSV)

    required_columns = {
        "domain",
        "source_type",
        "text",
    }

    missing_columns = required_columns - set(model_df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns in model CSV: {missing_columns}"
        )

    # Keep only actual model rows.
    model_df = model_df[
        model_df["source_type"] == "model"
    ].copy()

    # This gives the number of AI texts in each domain.
    required_per_domain = (
        model_df
        .groupby("domain")
        .size()
        .to_dict()
    )

    print("Required human texts per domain:")
    print(required_per_domain)

    human_rows = []

    for domain, number_required in sorted(
        required_per_domain.items()
    ):
        domain_rows = load_human_domain(
            domain=domain,
            number_required=int(number_required),
        )

        human_rows.extend(domain_rows)

    human_df = pd.DataFrame(human_rows)

    # Make human_df compatible with all columns in model_df.
    for column in model_df.columns:
        if column not in human_df.columns:
            human_df[column] = ""

    # Add the human_dataset metadata column to model rows.
    if "human_dataset" not in model_df.columns:
        model_df["human_dataset"] = ""

    # Use the same column order.
    human_df = human_df[model_df.columns]

    combined_df = pd.concat(
        [model_df, human_df],
        ignore_index=True,
    )

    human_df.to_csv(
        HUMAN_CSV,
        index=False,
    )

    combined_df.to_csv(
        COMBINED_CSV,
        index=False,
    )

    print("\nHuman dataset saved to:")
    print(HUMAN_CSV)

    print("\nCombined dataset saved to:")
    print(COMBINED_CSV)

    print("\nCounts by source type:")
    print(
        combined_df
        .groupby("source_type")
        .size()
    )

    print("\nCounts by domain and source type:")
    print(
        combined_df
        .groupby(["domain", "source_type"])
        .size()
    )

    print("\nHuman word-count summary:")
    print(
        human_df["n_words"].describe()
    )

    return human_df, combined_df

### Create dataset and Merge Csv


In [19]:
human_df, combined_df = create_human_dataset()

Required human texts per domain:
{'howto': 25, 'news': 25, 'opinion': 25, 'review': 25, 'story': 25}

--- Human texts: howto (25 required) ---
  Human text 1/25: 392 words
  Human text 2/25: 376 words
  Human text 3/25: 398 words
  Human text 4/25: 394 words
  Human text 5/25: 397 words
  Human text 6/25: 394 words
  Human text 7/25: 389 words
  Human text 8/25: 399 words
  Human text 9/25: 398 words
  Human text 10/25: 398 words
  Human text 11/25: 368 words
  Human text 12/25: 396 words
  Human text 13/25: 369 words
  Human text 14/25: 395 words
  Human text 15/25: 391 words
  Human text 16/25: 389 words
  Human text 17/25: 366 words
  Human text 18/25: 384 words
  Human text 19/25: 400 words
  Human text 20/25: 388 words
  Human text 21/25: 391 words
  Human text 22/25: 365 words
  Human text 23/25: 393 words
  Human text 24/25: 382 words
  Human text 25/25: 382 words

--- Human texts: news (25 required) ---
  Human text 1/25: 400 words
  Human text 2/25: 392 words
  Human text 3/25

Repo card metadata block was not found. Setting CardData to empty.


  Human text 1/25: 391 words
  Human text 2/25: 381 words
  Human text 3/25: 395 words
  Human text 4/25: 398 words
  Human text 5/25: 308 words
  Human text 6/25: 396 words
  Human text 7/25: 372 words
  Human text 8/25: 400 words
  Human text 9/25: 393 words
  Human text 10/25: 384 words
  Human text 11/25: 354 words
  Human text 12/25: 393 words
  Human text 13/25: 313 words
  Human text 14/25: 340 words
  Human text 15/25: 349 words
  Human text 16/25: 386 words
  Human text 17/25: 391 words
  Human text 18/25: 393 words
  Human text 19/25: 345 words
  Human text 20/25: 341 words
  Human text 21/25: 390 words
  Human text 22/25: 391 words
  Human text 23/25: 354 words
  Human text 24/25: 383 words
  Human text 25/25: 383 words

--- Human texts: review (25 required) ---


README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

  Human text 1/25: 386 words
  Human text 2/25: 391 words
  Human text 3/25: 301 words
  Human text 4/25: 301 words
  Human text 5/25: 318 words
  Human text 6/25: 341 words
  Human text 7/25: 311 words
  Human text 8/25: 400 words
  Human text 9/25: 352 words
  Human text 10/25: 366 words
  Human text 11/25: 398 words
  Human text 12/25: 381 words
  Human text 13/25: 376 words
  Human text 14/25: 302 words
  Human text 15/25: 395 words
  Human text 16/25: 315 words
  Human text 17/25: 392 words
  Human text 18/25: 352 words
  Human text 19/25: 320 words
  Human text 20/25: 352 words
  Human text 21/25: 385 words
  Human text 22/25: 394 words
  Human text 23/25: 339 words
  Human text 24/25: 379 words
  Human text 25/25: 396 words

--- Human texts: story (25 required) ---


README.md:   0%|          | 0.00/837 [00:00<?, ?B/s]

  Human text 1/25: 393 words
  Human text 2/25: 339 words
  Human text 3/25: 387 words
  Human text 4/25: 386 words
  Human text 5/25: 334 words
  Human text 6/25: 398 words
  Human text 7/25: 387 words
  Human text 8/25: 391 words
  Human text 9/25: 395 words
  Human text 10/25: 377 words
  Human text 11/25: 394 words
  Human text 12/25: 382 words
  Human text 13/25: 334 words
  Human text 14/25: 336 words
  Human text 15/25: 392 words
  Human text 16/25: 363 words
  Human text 17/25: 388 words
  Human text 18/25: 399 words
  Human text 19/25: 399 words
  Human text 20/25: 316 words
  Human text 21/25: 393 words
  Human text 22/25: 380 words
  Human text 23/25: 396 words
  Human text 24/25: 400 words
  Human text 25/25: 392 words

Human dataset saved to:
/content/drive/MyDrive/Final_Project_NLP/human_texts.csv

Combined dataset saved to:
/content/drive/MyDrive/Final_Project_NLP/dataset_with_humans.csv

Counts by source type:
source_type
human    125
model    125
dtype: int64

Counts b

### ECHANTILLONNAGE DES TEXTES HUMAINS (Inutile)

In [ ]:
def pick_text_column(example: dict, candidates) -> str:
    for c in candidates:
        if c in example and isinstance(example[c], str):
            return c
    # sinon : premiere colonne texte trouvee
    for k, v in example.items():
        if isinstance(v, str):
            return k
    raise ValueError("Aucune colonne texte trouvee : " + str(list(example.keys())))


def load_human_domain(domain: str) -> list:
    """Retourne une liste de textes humains propres et tronques pour un domaine."""
    print(f"\n--- Textes humains : {domain} ---")
    try:
        if domain == "news":
            ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="train", streaming=True)
            col_candidates = ["article", "text"]
        elif domain == "opinion":
            if not PERSUADE_CSV or not os.path.exists(PERSUADE_CSV):
                print("  (PERSUADE_CSV manquant -> domaine 'opinion' saute)")
                return []
            df = pd.read_csv(PERSUADE_CSV)
            df = df[df.get("generated", 0) == 0]
            texts = [clean_text(t) for t in df["text"].tolist()]
            texts = [truncate_words(t) for t in texts if n_words(t) >= MIN_WORDS_HUMAN]
            return texts[:N_HUMAN_PER_DOMAIN]
        elif domain == "review":
            ds = load_dataset("Yelp/yelp_review_full", split="train", streaming=True)
            col_candidates = ["text"]
        elif domain == "story":
            ds = load_dataset("euclaise/writingprompts", split="train", streaming=True)
            col_candidates = ["story", "text"]
        elif domain == "howto":
            ds = load_dataset("sentence-transformers/eli5", split="train", streaming=True)
            col_candidates = ["answer", "text", "response"]
        else:
            return []

        texts = []
        for ex in ds:                      # streaming = on ne telecharge pas tout
            col = pick_text_column(ex, col_candidates)
            t = clean_text(ex[col])
            if n_words(t) >= MIN_WORDS_HUMAN:
                texts.append(truncate_words(t))
            if len(texts) >= N_HUMAN_PER_DOMAIN:
                break
        print(f"  {len(texts)} textes retenus")
        return texts

    except Exception as e:
        print(f"  ERREUR sur {domain} : {e}")
        return []

